## HuggingFace Basics
## Pretrained Models & Pipelines
**Goal:** Learn to use pretrained models with HuggingFace Transformers

### Today's Plan:
1. Pipelines — one line NLP tasks
2. Tokenizers — text to numbers
3. Models — BERT, GPT-2, T5
4. Embeddings — sentence similarity
5. Build a mini NLP app

In [19]:
import nbformat

In [20]:
!pip install transformers datasets sentencepiece -q


In [21]:
# Imports
import torch
import numpy as np
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
)

print(f"Transformers ready!")
import transformers
print(f"Transformers version: {transformers.__version__}")
print(f"PyTorch version    : {torch.__version__}")
print(f"GPU available      : {torch.cuda.is_available()}")

Transformers ready!
Transformers version: 5.0.0
PyTorch version    : 2.10.0+cu128
GPU available      : True


## 🔥 Part 1 — Pipelines
The easiest way to use pretrained models.
One line of code = download + load + inference!

In [22]:
# Sentiment Analysis
print("Loading sentiment analysis pipeline...")
sentiment = pipeline(
    "sentiment-analysis",
    model = "distilbert-base-uncased-finetuned-sst-2-english"
)

# Test sentences
sentences = [
    "I absolutely love this movie! It was fantastic!",
    "This was the worst film I have ever seen.",
    "The movie was okay, nothing special.",
    "HuggingFace makes AI so easy to use!",
    "I am not sure how I feel about this.",
]

print("\nSentiment Analysis Results:")
print("="*55)
for sentence in sentences:
    result = sentiment(sentence)[0]
    label  = result['label']
    score  = result['score']
    emoji  = "😊" if label == "POSITIVE" else "😞"
    print(f"{emoji} {label} ({score:.4f})")
    print(f"   '{sentence[:50]}...'")
    print()

Loading sentiment analysis pipeline...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


Sentiment Analysis Results:
😊 POSITIVE (0.9999)
   'I absolutely love this movie! It was fantastic!...'

😞 NEGATIVE (0.9998)
   'This was the worst film I have ever seen....'

😞 NEGATIVE (0.9925)
   'The movie was okay, nothing special....'

😊 POSITIVE (0.9941)
   'HuggingFace makes AI so easy to use!...'

😞 NEGATIVE (0.9990)
   'I am not sure how I feel about this....'



In [23]:
# Text Generation with GPT-2
print("Loading GPT-2 text generation pipeline...")
generator = pipeline(
    "text-generation",
    model = "gpt2"
)

# Seed texts
seeds = [
    "Artificial intelligence will",
    "The future of machine learning is",
    "Once upon a time in a land far away",
]

print("\nGPT-2 Text Generation:")
print("="*55)
for seed in seeds:
    result = generator(
        seed,
        max_length      = 60,
        num_return_sequences = 1,
        temperature     = 0.8,
        do_sample       = True,
        pad_token_id    = 50256
    )[0]['generated_text']
    print(f"Seed  : '{seed}'")
    print(f"Output: {result}")
    print()

Loading GPT-2 text generation pipeline...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Both `max_new_tokens` (=256) and `max_length`(=60) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



GPT-2 Text Generation:


Both `max_new_tokens` (=256) and `max_length`(=60) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Seed  : 'Artificial intelligence will'
Output: Artificial intelligence will be a topic that will take up space during this year's CES and the panel's next event will be on the ARG Technology, an artificial intelligence technology that will allow us to interact with the human body without the need for a wheelchair.

"The ARG Technology is a way to create a better way to communicate with human bodies," said Robert DeNucci, head of the company's robotics division. "By letting you walk across a room using an augmented reality camera, it's possible to be more effective and more humane than you would have been if you had just had an AI robot."

What's more, the ARG Technology is an extension of the ARG-V, a form of real-time vision systems used to help individuals build more precise and reliable body representations. It's designed to interact with objects in real-time. It can do these things for you.

There are also some pretty cool features that are going to be part of this technology, incl

Both `max_new_tokens` (=256) and `max_length`(=60) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Seed  : 'The future of machine learning is'
Output: The future of machine learning is still young, but it will surely be a lot better than one that has been developed in the past. So far, there have been promising technologies for machine learning that have been available for some time now. In November, we saw the first demonstration of a multi-dimensional network for computing. This is a very interesting approach to training a machine, and in general is very fast and capable of generating some pretty good results. It has some limitations, but it is also very promising. We need more of these.

In terms of machine learning, there are many more applications: mapping data to machine learning, for instance. However, we still have a lot of work to do. For example, one of the main things we need to know for any future AI is how to identify that data. So, even though machines have the capabilities to learn and adapt to new situations, we still lack the information already. They are still lear

In [24]:
# Question Answering
print("Loading QA pipeline...")
qa = pipeline(
    "question-answering",
    model = "deepset/roberta-base-squad2"
)

# Context + Questions
context = """
HuggingFace was founded in 2016 by Clement Delangue,
Julien Chaumond, and Thomas Wolf. The company is
headquartered in New York City with offices in Paris.
HuggingFace is best known for its Transformers library
which provides thousands of pretrained models for NLP,
computer vision, and audio tasks. The company raised
$235 million in Series D funding in 2022 at a valuation
of $4.5 billion. HuggingFace has over 500,000 models
and 100,000 datasets on its platform.
"""

questions = [
    "When was HuggingFace founded?",
    "Where is HuggingFace headquartered?",
    "How much funding did HuggingFace raise?",
    "How many models are on HuggingFace?",
]

print("\nQuestion Answering Results:")
print("="*55)
for question in questions:
    result = qa(question=question, context=context)
    print(f"Q: {question}")
    print(f"A: {result['answer']} (score: {result['score']:.4f})")
    print()

Loading QA pipeline...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaForQuestionAnswering LOAD REPORT from: deepset/roberta-base-squad2
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Question Answering Results:
Q: When was HuggingFace founded?
A: 2016 (score: 0.9725)

Q: Where is HuggingFace headquartered?
A: New York City (score: 0.8692)

Q: How much funding did HuggingFace raise?
A: $235 million (score: 0.5613)

Q: How many models are on HuggingFace?
A: over 500,000 (score: 0.4387)



In [25]:
import transformers
print(transformers.__version__)

5.0.0


In [26]:
# Skip pipeline, use model directly instead
from transformers import BartForConditionalGeneration, BartTokenizer

print("Loading BART model...")
model_name = "facebook/bart-large-cnn"
tokenizer  = BartTokenizer.from_pretrained(model_name)
model      = BartForConditionalGeneration.from_pretrained(model_name)

article = """
Artificial intelligence has made remarkable progress
in recent years. Large language models like GPT-4 and
Claude have demonstrated unprecedented abilities in
natural language understanding and generation.
These models are trained on vast amounts of text data
using self-supervised learning techniques.
The transformer architecture, introduced in 2017,
has become the foundation of modern AI systems.
"""

# Tokenize
inputs = tokenizer(
    article,
    return_tensors  = "pt",
    max_length      = 1024,
    truncation      = True
)

# Generate summary
summary_ids = model.generate(
    inputs["input_ids"],
    max_length  = 60,
    min_length  = 20,
    length_penalty = 2.0,
    num_beams   = 4,
    early_stopping = True
)

# Decode
summary = tokenizer.decode(
    summary_ids[0],
    skip_special_tokens = True
)

print(f"Summary: {summary}")

Loading BART model...


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

Summary: The transformer architecture, introduced in 2017, has become the foundation of modern AI systems. Large language models like GPT-4 and Claude have demonstrated unprecedented abilities.


In [27]:
# ── Zero Shot Classification ──────────────────
# Classify text into categories WITHOUT training!
print("Loading zero-shot classification pipeline...")
classifier = pipeline(
    "zero-shot-classification",
    model = "facebook/bart-large-mnli"
)

texts = [
    "The stock market crashed today losing 500 points",
    "Scientists discover new treatment for cancer",
    "Manchester United won the championship 3-1",
    "The new iPhone has an amazing camera system",
]

# Labels we want to classify into
# Model has NEVER seen these labels before!
labels = ["finance", "health", "sports", "technology"]

print("\nZero-Shot Classification Results:")
print("(No training needed - model figures it out!)")
print("="*55)

for text in texts:
    result = classifier(text, candidate_labels=labels)
    top_label = result['labels'][0]
    top_score = result['scores'][0]
    print(f"Text  : '{text[:50]}...'")
    print(f"Label : {top_label} ({top_score:.4f})")
    print()

Loading zero-shot classification pipeline...


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]


Zero-Shot Classification Results:
(No training needed - model figures it out!)
Text  : 'The stock market crashed today losing 500 points...'
Label : finance (0.9709)

Text  : 'Scientists discover new treatment for cancer...'
Label : technology (0.8199)

Text  : 'Manchester United won the championship 3-1...'
Label : sports (0.9481)

Text  : 'The new iPhone has an amazing camera system...'
Label : technology (0.9810)



In [28]:
# ── Tokenizer Deep Dive ───────────────────────
print("="*55)
print("TOKENIZER DEEP DIVE")
print("="*55)

from transformers import AutoTokenizer

# Load BERT tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)

text = "HuggingFace makes AI easy!"

# ── Basic tokenization ────────────────────────
tokens     = tokenizer.tokenize(text)
input_ids  = tokenizer.encode(text)
decoded    = tokenizer.decode(input_ids)

print(f"\nOriginal text : '{text}'")
print(f"Tokens        : {tokens}")
print(f"Input IDs     : {input_ids}")
print(f"Decoded back  : '{decoded}'")
print(f"\nVocabulary size: {tokenizer.vocab_size:,}")

# ── Full tokenizer output ─────────────────────
print("\n" + "-"*55)
print("Full tokenizer output:")
inputs = tokenizer(
    text,
    return_tensors = "pt",
    padding        = True,
    truncation     = True,
    max_length     = 128
)
print(f"input_ids      : {inputs['input_ids']}")
print(f"attention_mask : {inputs['attention_mask']}")

# ── Special tokens ────────────────────────────
print("\n" + "-"*55)
print("Special Tokens:")
print(f"[CLS] token id : {tokenizer.cls_token_id}")
print(f"[SEP] token id : {tokenizer.sep_token_id}")
print(f"[PAD] token id : {tokenizer.pad_token_id}")
print(f"[UNK] token id : {tokenizer.unk_token_id}")

# ── Padding demo ──────────────────────────────
print("\n" + "-"*55)
print("Padding Demo (batch of different lengths):")

sentences = [
    "Short sentence",
    "This is a much longer sentence with more words"
]

batch = tokenizer(
    sentences,
    padding        = True,
    truncation     = True,
    max_length     = 20,
    return_tensors = "pt"
)

print(f"input_ids shape     : {batch['input_ids'].shape}")
print(f"Sentence 1 ids      : {batch['input_ids'][0]}")
print(f"Sentence 2 ids      : {batch['input_ids'][1]}")
print(f"Sentence 1 mask     : {batch['attention_mask'][0]}")
print(f"Sentence 2 mask     : {batch['attention_mask'][1]}")
print("(0 in mask = padding token = model ignores it)")

# ── Subword tokenization ──────────────────────
print("\n" + "-"*55)
print("Subword Tokenization (BPE):")
rare_words = [
    "unhappiness",
    "transformerization",
    "cryptocurrency",
    "backpropagation"
]

for word in rare_words:
    tokens = tokenizer.tokenize(word)
    print(f"  '{word}' → {tokens}")
print("(rare words split into subwords!)")

TOKENIZER DEEP DIVE

Original text : 'HuggingFace makes AI easy!'
Tokens        : ['hugging', '##face', 'makes', 'ai', 'easy', '!']
Input IDs     : [101, 17662, 12172, 3084, 9932, 3733, 999, 102]
Decoded back  : '[CLS] huggingface makes ai easy! [SEP]'

Vocabulary size: 30,522

-------------------------------------------------------
Full tokenizer output:
input_ids      : tensor([[  101, 17662, 12172,  3084,  9932,  3733,   999,   102]])
attention_mask : tensor([[1, 1, 1, 1, 1, 1, 1, 1]])

-------------------------------------------------------
Special Tokens:
[CLS] token id : 101
[SEP] token id : 102
[PAD] token id : 0
[UNK] token id : 100

-------------------------------------------------------
Padding Demo (batch of different lengths):
input_ids shape     : torch.Size([2, 11])
Sentence 1 ids      : tensor([ 101, 2460, 6251,  102,    0,    0,    0,    0,    0,    0,    0])
Sentence 2 ids      : tensor([ 101, 2023, 2003, 1037, 2172, 2936, 6251, 2007, 2062, 2616,  102])
Sentence 1 mask

In [29]:
# ── BERT Embeddings ───────────────────────────
print("="*55)
print("BERT EMBEDDINGS")
print("="*55)

from transformers import AutoModel
import torch

# Load BERT
print("\nLoading BERT...")
bert_tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)
bert_model = AutoModel.from_pretrained(
    "bert-base-uncased"
)
bert_model.eval()  # inference mode

def get_embeddings(texts):
    """
    Get sentence embeddings using [CLS] token.

    [CLS] token output = representation of
    entire sentence after attending to all words!
    """
    inputs  = bert_tokenizer(
        texts,
        padding        = True,
        truncation     = True,
        max_length     = 128,
        return_tensors = "pt"
    )

    with torch.no_grad():
        outputs = bert_model(**inputs)

    # [CLS] token = index 0 of last hidden state
    # Shape: (batch_size, hidden_size=768)
    cls_embeddings = outputs.last_hidden_state[:, 0, :]
    return cls_embeddings

# Test sentences
sentences = [
    "I love machine learning",
    "Deep learning is amazing",
    "The cat sat on the mat",
    "Dogs are great pets",
]

embeddings = get_embeddings(sentences)
print(f"\nEmbedding shape: {embeddings.shape}")
print(f"(batch_size=4, hidden_size=768)")
print(f"\nFirst embedding (first 5 dims):")
print(embeddings[0][:5])

BERT EMBEDDINGS

Loading BERT...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Embedding shape: torch.Size([4, 768])
(batch_size=4, hidden_size=768)

First embedding (first 5 dims):
tensor([ 0.1335,  0.2301, -0.0360, -0.0634, -0.1529])


In [30]:
# ── Sentence Similarity ───────────────────────
print("="*55)
print("SEMANTIC SIMILARITY WITH BERT")
print("="*55)

import torch.nn.functional as F

def cosine_similarity(emb1, emb2):
    """
    Cosine similarity between two embeddings.
    Range: -1 to 1
    1.0  = identical meaning
    0.0  = unrelated
    -1.0 = opposite meaning
    """
    return F.cosine_similarity(
        emb1.unsqueeze(0),
        emb2.unsqueeze(0)
    ).item()

# Test pairs
pairs = [
    ("I love pizza", "Pizza is my favorite food"),
    ("I love pizza", "The stock market crashed today"),
    ("Machine learning is fun", "Deep learning is exciting"),
    ("The cat is sleeping", "The dog is running"),
    ("How are you?", "How do you do?"),
]

print("\nSemantic Similarity Scores:")
print("-"*55)

for text1, text2 in pairs:
    emb1 = get_embeddings([text1])[0]
    emb2 = get_embeddings([text2])[0]
    sim  = cosine_similarity(emb1, emb2)

    # Visual bar
    bar = "█" * int(sim * 20)

    print(f"Text 1: '{text1}'")
    print(f"Text 2: '{text2}'")
    print(f"Score : {sim:.4f}  {bar}")
    print()

SEMANTIC SIMILARITY WITH BERT

Semantic Similarity Scores:
-------------------------------------------------------
Text 1: 'I love pizza'
Text 2: 'Pizza is my favorite food'
Score : 0.9790  ███████████████████

Text 1: 'I love pizza'
Text 2: 'The stock market crashed today'
Score : 0.9080  ██████████████████

Text 1: 'Machine learning is fun'
Text 2: 'Deep learning is exciting'
Score : 0.9581  ███████████████████

Text 1: 'The cat is sleeping'
Text 2: 'The dog is running'
Score : 0.9801  ███████████████████

Text 1: 'How are you?'
Text 2: 'How do you do?'
Score : 0.9484  ██████████████████



In [31]:
# ── Named Entity Recognition ──────────────────
print("="*55)
print("NAMED ENTITY RECOGNITION (NER)")
print("="*55)

ner = pipeline(
    "ner",
    model     = "dbmdz/bert-large-cased-finetuned-conll03-english",
    aggregation_strategy = "simple"
)

texts = [
    "Elon Musk founded Tesla and SpaceX in California.",
    "Barack Obama was born in Hawaii and studied at Harvard.",
    "Apple Inc was founded by Steve Jobs in Cupertino.",
    "The Eiffel Tower is located in Paris, France.",
]

print("\nNER Results:")
print("-"*55)

for text in texts:
    print(f"\nText: '{text}'")
    entities = ner(text)
    for entity in entities:
        print(f"  [{entity['entity_group']}] "
              f"'{entity['word']}' "
              f"(score: {entity['score']:.3f})")

NAMED ENTITY RECOGNITION (NER)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



NER Results:
-------------------------------------------------------

Text: 'Elon Musk founded Tesla and SpaceX in California.'
  [PER] 'Elon Musk' (score: 0.997)
  [ORG] 'Tesla' (score: 0.990)
  [ORG] 'SpaceX' (score: 0.998)
  [LOC] 'California' (score: 1.000)

Text: 'Barack Obama was born in Hawaii and studied at Harvard.'
  [PER] 'Barack Obama' (score: 0.999)
  [LOC] 'Hawaii' (score: 0.999)
  [ORG] 'Harvard' (score: 0.986)

Text: 'Apple Inc was founded by Steve Jobs in Cupertino.'
  [ORG] 'Apple Inc' (score: 1.000)
  [PER] 'Steve Jobs' (score: 0.982)
  [LOC] 'Cupertino' (score: 0.969)

Text: 'The Eiffel Tower is located in Paris, France.'
  [MISC] 'Eiffel Tower' (score: 0.694)
  [LOC] 'Paris' (score: 0.999)
  [LOC] 'France' (score: 0.999)


In [32]:
# ── Fill Mask ─────────────────────────────────
print("="*55)
print("FILL MASK — BERT PREDICTS MISSING WORDS")
print("="*55)

fill_mask = pipeline(
    "fill-mask",
    model = "bert-base-uncased"
)

# Sentences with [MASK] token
masked_sentences = [
    "Paris is the [MASK] of France.",
    "The [MASK] is the powerhouse of the cell.",
    "Python is a popular [MASK] language.",
    "The sun rises in the [MASK].",
    "HuggingFace is the [MASK] of AI models.",
]

print("\nBERT Fill Mask Predictions:")
print("-"*55)

for sentence in masked_sentences:
    results = fill_mask(sentence)
    print(f"\nSentence: '{sentence}'")
    print("Top 3 predictions:")
    for i, result in enumerate(results[:3]):
        token = result['token_str']
        score = result['score']
        print(f"  {i+1}. '{token}' ({score:.4f})")

FILL MASK — BERT PREDICTS MISSING WORDS


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



BERT Fill Mask Predictions:
-------------------------------------------------------

Sentence: 'Paris is the [MASK] of France.'
Top 3 predictions:
  1. 'capital' (0.9969)
  2. 'heart' (0.0006)
  3. 'center' (0.0004)

Sentence: 'The [MASK] is the powerhouse of the cell.'
Top 3 predictions:
  1. 'female' (0.0193)
  2. 'woman' (0.0143)
  3. 'speaker' (0.0110)

Sentence: 'Python is a popular [MASK] language.'
Top 3 predictions:
  1. 'programming' (0.9610)
  2. 'python' (0.0062)
  3. 'natural' (0.0046)

Sentence: 'The sun rises in the [MASK].'
Top 3 predictions:
  1. 'sky' (0.6662)
  2. 'west' (0.1048)
  3. 'east' (0.0950)

Sentence: 'HuggingFace is the [MASK] of AI models.'
Top 3 predictions:
  1. 'name' (0.1585)
  2. 'color' (0.0338)
  3. 'application' (0.0278)


In [33]:
# ── Mini NLP App ──────────────────────────────
print("="*55)
print("MINI NLP APP — PUTTING IT ALL TOGETHER")
print("="*55)

def analyze_text(text):
    """
    Complete NLP analysis of any text:
    1. Sentiment
    2. Named Entities
    3. Key words (fill mask style)
    """
    print(f"\nAnalyzing: '{text}'")
    print("-"*50)

    # 1. Sentiment
    sent_result = sentiment(text)[0]
    print(f"Sentiment : {sent_result['label']} "
          f"({sent_result['score']:.3f})")

    # 2. Named Entities
    entities = ner(text)
    if entities:
        entity_str = ", ".join([
            f"{e['word']} ({e['entity_group']})"
            for e in entities
        ])
        print(f"Entities  : {entity_str}")
    else:
        print(f"Entities  : None found")

    # 3. Embedding
    emb = get_embeddings([text])[0]
    print(f"Embedding : shape={emb.shape}, "
          f"mean={emb.mean():.4f}")

    print()

# Test our mini app
test_texts = [
    "Elon Musk announced that Tesla will launch a new electric car in 2025.",
    "I am really disappointed with the poor customer service at Amazon.",
    "Google DeepMind published an amazing research paper on AI safety.",
    "The weather in Mumbai today is very hot and humid.",
]

for text in test_texts:
    analyze_text(text)

MINI NLP APP — PUTTING IT ALL TOGETHER

Analyzing: 'Elon Musk announced that Tesla will launch a new electric car in 2025.'
--------------------------------------------------
Sentiment : POSITIVE (0.998)
Entities  : Elon Musk (PER), Tesla (ORG)
Embedding : shape=torch.Size([768]), mean=-0.0114


Analyzing: 'I am really disappointed with the poor customer service at Amazon.'
--------------------------------------------------
Sentiment : NEGATIVE (1.000)
Entities  : Amazon (ORG)
Embedding : shape=torch.Size([768]), mean=-0.0110


Analyzing: 'Google DeepMind published an amazing research paper on AI safety.'
--------------------------------------------------
Sentiment : POSITIVE (1.000)
Entities  : Google DeepMind (ORG)
Embedding : shape=torch.Size([768]), mean=-0.0113


Analyzing: 'The weather in Mumbai today is very hot and humid.'
--------------------------------------------------
Sentiment : POSITIVE (0.999)
Entities  : Mumbai (LOC)
Embedding : shape=torch.Size([768]), mean=-0.0108



## HuggingFace Basics Complete!

### What We Covered:
| Topic | Model Used | Key Learning |
|-------|-----------|--------------|
| Sentiment Analysis | DistilBERT | pipeline() magic |
| Text Generation | GPT-2 | temperature effect |
| Zero-Shot Classification | BART | no training needed! |
| Question Answering | RoBERTa | context + question |
| Tokenization | BERT | subword tokenization |
| Embeddings | BERT | [CLS] = sentence vector |
| Similarity | BERT | cosine similarity |
| NER | BERT-large | entity extraction |
| Fill Mask | BERT | masked language model |

### Key Takeaways:
1. pipeline() = one line to use any model
2. AutoClasses work for ANY model
3. [CLS] token = sentence representation
4. Subword tokenization handles rare words
5. Embeddings enable semantic search
6. BERT = understanding, GPT = generation